# Module 10, Part 5: Responsible deployment, bias auditing and model cards

This notebook is the runnable companion to Part 5 of the Module 10 learning content. A single headline metric can hide who a model fails. On a synthetic lending data set, where we know the truth, we run a bias audit end to end.

1. **Build the data and train a model** on biased historical decisions, with the protected attribute *not* given to the model.
2. **Audit it**: selection rate, true and false positive rates by group, and the gaps that define demographic parity and equalized odds.
3. **Find the leak**: a proxy feature (neighbourhood) that carries the group, and what happens when it is removed.
4. **Compare three fixes**: train on the true outcome, drop the proxy, or set group-specific thresholds. See what each costs.
5. **Write a model card** from the results, in code, so it is generated and not remembered.

**Setting.** 8,000 synthetic applicants, 40% in group 1. Real repayment depends only on a hidden creditworthiness that has the *same* distribution in both groups. The historical approvals we train on were biased against group 1. The first 5,000 rows are training data, the last 3,000 are the audit set. Group is used only to audit.

In [1]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
INK, BODY, MUTED = "#14161B", "#4E5563", "#7C838C"
NAVY, GREEN, AMBER, KEY, RULE = "#1C3D6E", "#0E6B57", "#8A6A18", "#7A2E8A", "#D9DCE2"

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": RULE, "axes.labelcolor": BODY, "xtick.color": BODY, "ytick.color": BODY,
    "axes.grid": True, "grid.color": RULE, "font.size": 10,
})

In [2]:
import numpy as np
def lending_data(n=6000, seed=0, label_bias=1.0, proxy_strength=0.8):
    """A synthetic loan data set in which the truth is known.

    Each applicant has a hidden creditworthiness, a group (0 or 1), and observed features. The group is
    NOT given to the model as a feature, but `neighbourhood` is a proxy for it. Historical approvals
    (the label we train on) were biased against group 1 by `label_bias` units of credit score, whereas
    the real repayment outcome depends only on creditworthiness.
    Returns a dict of numpy arrays: X, group, y_hist (biased label), y_true (repayment), names.
    """
    rng = np.random.default_rng(seed)
    group = (rng.random(n) < 0.4).astype(int)
    credit = rng.normal(0.0, 1.0, n)                        # same distribution in both groups
    income = 0.8 * credit + rng.normal(0, 0.6, n)
    history = 0.7 * credit + rng.normal(0, 0.7, n)
    debt = -0.5 * credit + rng.normal(0, 0.8, n)
    noise = rng.normal(0, 1, n)
    neighbourhood = proxy_strength * (group - 0.4) / 0.49 + np.sqrt(1 - proxy_strength ** 2) * rng.normal(0, 1, n)
    X = np.stack([income, history, debt, neighbourhood, noise], 1).astype(np.float32)
    y_true = (credit + rng.normal(0, 0.5, n) > 0).astype(int)           # repays the loan
    y_hist = (credit - label_bias * group + rng.normal(0, 0.5, n) > 0).astype(int)  # biased past approvals
    return dict(X=X, group=group, y_hist=y_hist, y_true=y_true,
                names=["income", "payment history", "debt", "neighbourhood", "noise"])

In [3]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def fit_logistic(X, y, epochs=400, lr=0.05, seed=0, l2=1e-3):
    torch.manual_seed(seed)
    X = torch.as_tensor(X, dtype=torch.float32); y = torch.as_tensor(y, dtype=torch.float32)
    mu, sd = X.mean(0), X.std(0) + 1e-6
    lin = nn.Linear(X.shape[1], 1)
    opt = torch.optim.Adam(lin.parameters(), lr=lr)
    for _ in range(epochs):
        loss = F.binary_cross_entropy_with_logits(lin(((X - mu) / sd)).squeeze(1), y) + l2 * (lin.weight ** 2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    def predict_proba(Z):
        Z = torch.as_tensor(Z, dtype=torch.float32)
        with torch.no_grad(): return torch.sigmoid(lin(((Z - mu) / sd))).squeeze(1).numpy()
    predict_proba.coef = (lin.weight.detach() / sd).squeeze(0).numpy()
    return predict_proba


def group_rates(pred, y, group):
    """Selection rate, true positive rate, false positive rate and accuracy per group."""
    out = {}
    for g in np.unique(group):
        m = group == g
        p, t = pred[m], y[m]
        out[int(g)] = dict(n=int(m.sum()), selection=float(p.mean()),
                           tpr=float(p[t == 1].mean()) if (t == 1).any() else float("nan"),
                           fpr=float(p[t == 0].mean()) if (t == 0).any() else float("nan"),
                           accuracy=float((p == t).mean()))
    return out


def gaps(rates):
    a, b = rates[0], rates[1]
    return dict(demographic_parity_gap=abs(a["selection"] - b["selection"]),
                tpr_gap=abs(a["tpr"] - b["tpr"]), fpr_gap=abs(a["fpr"] - b["fpr"]),
                disparate_impact_ratio=min(a["selection"], b["selection"]) / max(a["selection"], b["selection"]))


def proxy_leakage(X, group, col, seed=0):
    """How well one column predicts the protected group (accuracy of a threshold rule, 0.5 = no leakage with balanced groups)."""
    x = X[:, col]; best = 0.0
    for t in np.quantile(x, np.linspace(0.05, 0.95, 37)):
        acc = max(((x > t) == group).mean(), ((x <= t) == group).mean()); best = max(best, acc)
    base = max(group.mean(), 1 - group.mean())
    return float(best), float(base)


def threshold_for_rate(scores, rate):
    return float(np.quantile(scores, 1 - rate))


def per_group_thresholds(scores, group, target_selection):
    """Group-specific thresholds that give each group the same selection rate (a post-processing mitigation)."""
    return {int(g): threshold_for_rate(scores[group == g], target_selection) for g in np.unique(group)}


def apply_thresholds(scores, group, th):
    return np.array([scores[i] > th[int(group[i])] for i in range(len(scores))]).astype(int)


def model_card(name, intended_use, data, metrics, subgroup, limitations, extra=None):
    """Return the model card as a Markdown string. `metrics` and `subgroup` are plain dicts of numbers."""
    L = [f"# Model card: {name}", "", "## Intended use", intended_use, "", "## Data", data, "", "## Headline metrics"]
    for k, v in metrics.items(): L.append(f"- {k}: {v:.3f}" if isinstance(v, float) else f"- {k}: {v}")
    L += ["", "## Results by subgroup"]
    for g, r in subgroup.items():
        L.append(f"- group {g}: " + ", ".join(f"{k} {v:.3f}" if isinstance(v, float) else f"{k} {v}" for k, v in r.items()))
    L += ["", "## Known limitations", limitations]
    if extra:
        L += ["", "## Notes", extra]
    return "\n".join(L) + "\n"

In [4]:
D = lending_data(8000, 0)
X, g, yh, yt, names = D["X"], D["group"], D["y_hist"], D["y_true"], D["names"]
tr, te = slice(0, 5000), slice(5000, 8000)
print("group 1 share:", round(g.mean(), 3))
print("historical approval rate, group 0 / group 1:", round(yh[g == 0].mean(), 3), "/", round(yh[g == 1].mean(), 3))
print("true repayment rate,      group 0 / group 1:", round(yt[g == 0].mean(), 3), "/", round(yt[g == 1].mean(), 3))

group 1 share: 0.403
historical approval rate, group 0 / group 1: 0.496 / 0.199
true repayment rate,      group 0 / group 1: 0.502 / 0.523


The two groups repay at the same rate, but group 1 was approved much less often in the past. That is the label the model will learn from: a record of past decisions, not of past outcomes.

## 1 and 2. Train on history and audit

In [5]:
f = fit_logistic(X[tr], yh[tr]); s = f(X[te]); pred = (s > 0.5).astype(int)
r = group_rates(pred, yt[te], g[te]); G = gaps(r)
print(f"headline accuracy against real repayment: {(pred == yt[te]).mean():.3f}")
print(f"{'group':>6s} {'n':>6s} {'selection':>10s} {'TPR':>6s} {'FPR':>6s} {'accuracy':>9s}")
for k, v in r.items(): print(f"{k:6d} {v['n']:6d} {v['selection']:10.3f} {v['tpr']:6.3f} {v['fpr']:6.3f} {v['accuracy']:9.3f}")
print({k: round(v, 3) for k, v in G.items()})
print("model weight per raw unit of each feature:", {n: round(float(c), 2) for n, c in zip(names, f.coef)})

headline accuracy against real repayment: 0.720
 group      n  selection    TPR    FPR  accuracy
     0   1785      0.438  0.697  0.173     0.761
     1   1215      0.204  0.368  0.027     0.658
{'demographic_parity_gap': 0.234, 'tpr_gap': 0.329, 'fpr_gap': 0.145, 'disparate_impact_ratio': 0.466}
model weight per raw unit of each feature: {'income': 1.18, 'payment history': 0.73, 'debt': -0.38, 'neighbourhood': -0.84, 'noise': -0.07}


A single accuracy number looks acceptable, and hides a large gap. Group 1 is selected at less than half the rate of group 0 (the **disparate impact ratio** is far below the often-quoted 0.8 threshold), and among applicants who *would have repaid*, group 1 is approved much less often (the true positive rate gap). Both are **error rates that differ by group**, which is what equalized odds asks us to look at, and the model never saw the group column.

## 3. Where did the bias come from?

The model has no group feature, but it has `neighbourhood`, which we built to correlate with the group. The negative coefficient on it above is the mechanism: the model learned that the neighbourhood predicts a rejection, because in the history it did.

In [6]:
acc, base_rate = proxy_leakage(X, g, names.index("neighbourhood"))
print(f"a single threshold on neighbourhood predicts the group with accuracy {acc:.3f}  (always guessing the larger group: {base_rate:.3f})")
for j, n in enumerate(names): print(f"  {n:16s} best single-threshold accuracy at predicting group: {proxy_leakage(X, g, j)[0]:.3f}")
keep = [i for i, n in enumerate(names) if n != "neighbourhood"]
f2 = fit_logistic(X[tr][:, keep], yh[tr]); s2 = f2(X[te][:, keep]); r2 = group_rates((s2 > 0.5).astype(int), yt[te], g[te])
print()
print("without the neighbourhood column:", {k: round(v, 3) for k, v in gaps(r2).items()})

a single threshold on neighbourhood predicts the group with accuracy 0.915  (always guessing the larger group: 0.597)
  income           best single-threshold accuracy at predicting group: 0.591
  payment history  best single-threshold accuracy at predicting group: 0.590
  debt             best single-threshold accuracy at predicting group: 0.590
  neighbourhood    best single-threshold accuracy at predicting group: 0.915
  noise            best single-threshold accuracy at predicting group: 0.589

without the neighbourhood column: {'demographic_parity_gap': 0.023, 'tpr_gap': 0.031, 'fpr_gap': 0.002, 'disparate_impact_ratio': 0.931}


Neighbourhood predicts group membership far better than the other features, which carry almost no information about it. Removing that single column nearly closes the gap, which shows the mechanism, but it is **not a general recipe**: it worked here because the proxy was the *only* channel and we knew which column it was. Real data have many weak proxies (postcode, name, shopping history), and "fairness through unawareness", simply leaving the protected attribute out, is not a safeguard. You find leaks by auditing outcomes by group, not by inspecting the feature list.

## 4. Three fixes and what each one costs

In [7]:
rows = []
def report(name, pred_te):
    rr = group_rates(pred_te, yt[te], g[te]); gg = gaps(rr)
    rows.append((name, (pred_te == yt[te]).mean(), gg["demographic_parity_gap"], gg["tpr_gap"], gg["fpr_gap"], gg["disparate_impact_ratio"]))
report("trained on history (baseline)", pred)
report("drop the proxy column", (s2 > 0.5).astype(int))
f3 = fit_logistic(X[tr], yt[tr]); report("train on real repayment", (f3(X[te]) > 0.5).astype(int))
th = per_group_thresholds(s, g[te], pred.mean()); report("group thresholds, same selection rate", apply_thresholds(s, g[te], th))
print(f"{'fix':40s} {'accuracy':>9s} {'parity gap':>11s} {'TPR gap':>8s} {'FPR gap':>8s} {'impact ratio':>13s}")
for r_ in rows: print(f"{r_[0]:40s} {r_[1]:9.3f} {r_[2]:11.3f} {r_[3]:8.3f} {r_[4]:8.3f} {r_[5]:13.3f}")
print("thresholds by group:", {k: round(v, 3) for k, v in th.items()})

fix                                       accuracy  parity gap  TPR gap  FPR gap  impact ratio
trained on history (baseline)                0.720       0.234    0.329    0.145         0.466
drop the proxy column                        0.739       0.023    0.031    0.002         0.931
train on real repayment                      0.779       0.013    0.009    0.004         0.974
group thresholds, same selection rate        0.748       0.000    0.004    0.009         0.999
thresholds by group: {0: 0.613, 1: 0.304}


All three reduce the gaps, and none costs accuracy *against the real outcome*: accuracy actually rises, because the baseline's errors came from copying a biased label. Training on the real repayment is the best fix when it is available, but in practice the real outcome is observed only for the people who were approved, so you rarely have it for the rejected group. Group-specific thresholds need no retraining, but they use the protected attribute at decision time, which may be legally restricted, and they equalize the selection rate by construction, not the error rates. There is no single fix, and the three fairness criteria (equal selection rates, equal error rates, equal calibration) cannot in general all hold at once when the groups differ in anything but the label. Choosing between them is a decision for the people affected and the organization, not a coding step.

## 5. A model card, generated from the audit

In [8]:
card = model_card(
    name="Loan approval model v1 (synthetic, for teaching)",
    intended_use="Rank loan applicants by estimated repayment probability. Not for use on real applicants. A human reviews every rejection.",
    data="8,000 synthetic applicants, 5,000 for training, 3,000 for audit. Training labels: historical approvals, which were biased against group 1. Features: income, payment history, debt, neighbourhood, noise.",
    metrics={"accuracy against real repayment": float((pred == yt[te]).mean()), "audit set size": 3000},
    subgroup={k: dict(v) for k, v in r.items()},
    limitations="Trained on biased historical decisions. Selection rate and true positive rate differ strongly by group. The neighbourhood feature acts as a proxy for group. Mitigations tested: dropping the proxy, training on repayment outcomes, group thresholds. See the audit table for each.",
    extra=f"Disparate impact ratio {G['disparate_impact_ratio']:.2f}; demographic parity gap {G['demographic_parity_gap']:.2f}; TPR gap {G['tpr_gap']:.2f}; FPR gap {G['fpr_gap']:.2f}.")
print(card)

# Model card: Loan approval model v1 (synthetic, for teaching)

## Intended use
Rank loan applicants by estimated repayment probability. Not for use on real applicants. A human reviews every rejection.

## Data
8,000 synthetic applicants, 5,000 for training, 3,000 for audit. Training labels: historical approvals, which were biased against group 1. Features: income, payment history, debt, neighbourhood, noise.

## Headline metrics
- accuracy against real repayment: 0.720
- audit set size: 3000

## Results by subgroup
- group 0: n 1785, selection 0.438, tpr 0.697, fpr 0.173, accuracy 0.761
- group 1: n 1215, selection 0.204, tpr 0.368, fpr 0.027, accuracy 0.658

## Known limitations
Trained on biased historical decisions. Selection rate and true positive rate differ strongly by group. The neighbourhood feature acts as a proxy for group. Mitigations tested: dropping the proxy, training on repayment outcomes, group thresholds. See the audit table for each.

## Notes
Disparate impact ratio 

A model card is useful only if it states what the model *should not* be used for and shows the results **by subgroup**, not only the headline number. Generating it from the audit code means the numbers cannot drift from the evidence. Lab 10 asks for a short card for your own compressed model: what it is for, the data, the accuracy and the cost numbers, how it behaves on at least one slice, and what you do not know.

**Try it.** Set `label_bias=0` in `lending_data` and confirm the audit finds no gap. Raise `proxy_strength` to 0.95 and see whether dropping the column still works. Add a second, weaker proxy and see what the audit reveals.